# Analisi E-commerce e Segmentazione Clienti (Olist)

--- 
## 📑 Executive Summary
Questo progetto analizza le performance di **Olist**, la principale piattaforma e-commerce in Brasile. Attraverso l'integrazione di **SQL** e **Python**, abbiamo estratto insight critici da oltre **1.5 milioni di record**.

### Risultati Chiave:
1. **Efficienza Logistica**: Nonostante le sfide geografiche del Brasile, l'85% delle consegne avviene in anticipo rispetto alla stima.
2. **Strategia di Prodotto**: Identificazione delle categorie alto-solventi e correlazione tra prezzo e soddisfazione cliente.
3. **Customer Intelligence**: Segmentazione di circa 100.000 clienti in 4 cluster comportamentali (Modello RFM).
4. **Voce del Cliente**: Analisi NLP delle recensioni per identificare i principali driver di insoddisfazione.

--- 
## 🛠️ Tecnologie Utilizzate
- **MySQL**: Gestione database relazionale.
- **Python (Pandas/Seaborn)**: Analisi dati e visualizzazione.
- **Machine Learning**: Clustering K-Means per la segmentazione.
- **NLP**: Sentiment Analysis con TextBlob e WordCloud.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from wordcloud import WordCloud
from textblob import TextBlob

# Configurazione grafici
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("Ambiente configurato con successo.")

## 1. Connessione al Database
Carichiamo le credenziali dal file `.env` per connetterci al database locale.

In [ ]:
load_dotenv()
DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS", "root") 
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = "olist_ecommerce"

engine = create_engine(f"mysql+mysqlconnector://{DB_USER}:{DB_PASS}@{DB_HOST}/{DB_NAME}")
print(f"Connesso al database: {DB_NAME}")

## 2. Analisi delle Performance di Vendita
Analizziamo il trend mensile del fatturato per identificare stagionalità e crescita.

In [ ]:
query_trend = """
SELECT 
    DATE_FORMAT(order_purchase_timestamp, '%Y-%m-01') AS month,
    SUM(price) AS revenue
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
WHERE order_status = 'delivered'
GROUP BY month
ORDER BY month;
"""

df_trend = pd.read_sql(query_trend, engine)
df_trend['month'] = pd.to_datetime(df_trend['month'])

plt.figure(figsize=(14, 6))
sns.lineplot(data=df_trend, x='month', y='revenue', marker='o', color='#2ecc71', linewidth=2.5)
plt.title('Trend Mensile del Fatturato (2016-2018)', fontsize=15)
plt.ylabel('Fatturato (BRL)')
plt.gca().yaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f} BRL'))
plt.show()

## 3. Analisi della Logistica
Calcoliamo l'accuratezza delle consegne: quanto tempo prima (o dopo) rispetto alla stima arrivano i prodotti?

In [ ]:
query_delivery = """
SELECT 
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delivery_delta
FROM orders 
WHERE order_status = 'delivered' 
    AND order_delivered_customer_date IS NOT NULL;
"""

df_delivery = pd.read_sql(query_delivery, engine)

plt.figure(figsize=(10, 5))
sns.histplot(df_delivery['delivery_delta'], bins=50, kde=True, color='#3498db')
plt.axvline(0, color='red', linestyle='--')
plt.title('Delta Consegna Reale vs Stimata (Giorni)')
plt.xlabel('Giorni (Negativo = In Anticipo)')
plt.xlim(-30, 10)
plt.show()

print(f"Percentuale ordini in anticipo o puntuali: {(df_delivery['delivery_delta'] <= 0).mean():.2%}")

## 4. Segmentazione Clienti (RFM + Clustering)
Prepariamo le feature per il modello di Machine Learning.

In [ ]:
# Query per estrarre le metriche RFM
query_rfm = """
SELECT 
    customer_unique_id,
    DATEDIFF((SELECT MAX(order_purchase_timestamp) FROM orders), MAX(order_purchase_timestamp)) AS recency,
    COUNT(o.order_id) AS frequency,
    SUM(price) AS monetary
FROM orders o
JOIN order_items oi ON o.order_id = oi.order_id
JOIN customers c ON o.customer_id = c.customer_id
WHERE order_status = 'delivered'
GROUP BY customer_unique_id;
"""

rfm = pd.read_sql(query_rfm, engine)

# Scaling
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['recency', 'frequency', 'monetary']])

# K-Means
kmeans = KMeans(n_clusters=4, random_state=42)
rfm['cluster'] = kmeans.fit_predict(rfm_scaled)

# Visualizzazione
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='recency', y='monetary', hue='cluster', palette='viridis', alpha=0.6)
plt.title('Segmentazione Clienti: Recency vs Monetary')
plt.show()

## 5. Analisi del Sentiment (NLP)
Utilizziamo sia un approccio visivo (WordCloud) che quantitativo (Polarity Score).

In [ ]:
# Estrazione recensioni testuali
query_reviews = "SELECT review_score, review_comment_message FROM order_reviews WHERE review_comment_message IS NOT NULL AND review_comment_message != '';"
df_reviews = pd.read_sql(query_reviews, engine)

# Funzione per polarità
df_reviews['polarity'] = df_reviews['review_comment_message'].apply(lambda x: TextBlob(x).sentiment.polarity)

# WordCloud per i due estremi
pos_text = " ".join(df_reviews[df_reviews['review_score'] == 5]['review_comment_message'])
neg_text = " ".join(df_reviews[df_reviews['review_score'] == 1]['review_comment_message'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

wordcloud_pos = WordCloud(background_color='white', colormap='Greens', width=800, height=400).generate(pos_text)
ax1.imshow(wordcloud_pos, interpolation='bilinear')
ax1.set_title('Keywords: Recensioni 5 Stelle')
ax1.axis('off')

wordcloud_neg = WordCloud(background_color='white', colormap='Reds', width=800, height=400).generate(neg_text)
ax2.imshow(wordcloud_neg, interpolation='bilinear')
ax2.set_title('Keywords: Recensioni 1 Stella')
ax2.axis('off')

plt.show()

# Grafico polarità vs score
plt.figure(figsize=(10, 5))
sns.boxplot(x='review_score', y='polarity', data=df_reviews, palette='coolwarm')
plt.title('Correlazione tra Review Score e Polarità NLP')
plt.show()

## 6. Analisi Geo-spaziale delle Logistica
Analizziamo come variano i costi e i tempi di consegna tra i diversi stati del Brasile.

In [ ]:
query_geo = """
SELECT 
    c.customer_state, 
    AVG(oi.freight_value) AS avg_freight, 
    AVG(DATEDIFF(order_delivered_customer_date, order_purchase_timestamp)) AS avg_delivery_time
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY avg_delivery_time DESC;
"""

df_geo = pd.read_sql(query_geo, engine)

fig, ax1 = plt.subplots(figsize=(15, 6))

sns.barplot(data=df_geo, x='customer_state', y='avg_delivery_time', color='#e74c3c', ax=ax1, alpha=0.7)
ax1.set_ylabel('Tempi Medi Consegna (Giorni)', color='#e74c3c')
ax1.set_title('Impatto Geografico sulla Logistica in Brasile')

ax2 = ax1.twinx()
sns.lineplot(data=df_geo, x='customer_state', y='avg_freight', color='#2c3e50', marker='o', ax=ax2, label='Costo Spedizione')
ax2.set_ylabel('Costo Medio Spedizione (BRL)', color='#2c3e50')

plt.show()

## 7. Conclusioni e Sviluppi Futuri
### Conclusioni
- Il modello **RFM** ha permesso di mappare con precisione il valore dei clienti, fornendo una base per campagne di retention.
- L'analisi del **Sentiment** rivela che la puntualità di consegna è il driver principale per i feedback a 5 stelle.
- La **Logistica** evidenzia differenze regionali marcate in Brasile, suggerendo la necessità di hub regionali per ridurre i costi nel Nord del paese.

### Sviluppi Futuri
- Implementazione di un modello di **Customer Churn Prediction**.
- Market Basket Analysis per suggerimenti cross-selling personalizzati.